# 04 — Modelo Clásico 2: BiLSTM

**Autora:** Yibby González  
**Proyecto:** Aprendizaje Profundo · Maestría en IA · Pontificia Universidad Javeriana · 2026  
**Dataset:** Andalusian Hotels Reviews (5 clases, desbalanceado)  
**Arquitectura:** Embedding → BiLSTM (bidireccional con dropout) → Dense → Softmax  

---

## Objetivo

Implementar el Modelo Clásico 2 del Proyecto Principal: una red BiLSTM bidireccional
para clasificación de reseñas de hoteles en 5 clases (1–5 estrellas).

La BiLSTM lee la secuencia en **ambas direcciones** (izquierda→derecha y derecha→izquierda),
capturando contexto futuro y pasado simultáneamente. Esto le da mayor capacidad representacional
que el LSTM unidireccional de Felipe, lo que hace la comparación más interesante.

## Experimentos planificados

| Experimento | Descripción | Propósito |
|---|---|---|
| `bilstm_v1` | BiLSTM base sin class weights | Línea base |
| `bilstm_v2` | BiLSTM con class weights | Manejar desbalance |
| `bilstm_v3` (opcional) | BiLSTM con embeddings pre-cargados FastText | Comparar embeddings |

> **Regla del equipo:** Iterar hiperparámetros solo en val set. El test set se mira **una sola vez**
> al final, una vez elegida la mejor configuración.

## 0. Configuración global

In [ ]:
# ── Configuración global ──────────────────────────────────────────────────────
import random, os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Semilla fija del equipo — NO modificar
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths comunes
ROOT        = Path("..")
DATA_DIR    = ROOT / "data"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# Importar módulos del equipo
sys.path.insert(0, str(ROOT))
from src.preprocessing import build_pipeline
from src.training      import train, evaluate, set_seed
from src.metrics       import finalize_and_save

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Semilla fija  : {SEED}")
print(f"Dispositivo   : {DEVICE}")
print(f"PyTorch       : {torch.__version__}")


## 1. Cargar datos con el pipeline de preprocesamiento

In [ ]:
# ── Cargar pipeline ──────────────────────────────────────────────────────────
# Ajustar text_col y label_col según el CSV del dataset
PIPELINE_CONFIG = {
    "data_dir"  : str(DATA_DIR),
    "text_col"  : "review",    # ajustar si difiere
    "label_col" : "rating",    # ajustar si difiere
    "vocab_size": 20_000,
    "max_len"   : 200,         # actualizar con percentil 95 del EDA de Felipe
    "batch_size": 64,
}

pipeline = build_pipeline(PIPELINE_CONFIG)

train_loader   = pipeline["train_loader"]
val_loader     = pipeline["val_loader"]
test_loader    = pipeline["test_loader"]
word2idx       = pipeline["word2idx"]
class_weights  = pipeline["class_weights"]
VOCAB_SIZE     = pipeline["vocab_size"]
MAX_LEN        = pipeline["max_len"]
N_CLASSES      = 5       # 1-5 estrellas → 0-4 (0-indexed)
CLASS_NAMES    = ["1", "2", "3", "4", "5"]

print(f"\nDataset listo:")
print(f"  Vocab size  : {VOCAB_SIZE:,}")
print(f"  max_len     : {MAX_LEN}")
print(f"  N clases    : {N_CLASSES}")
print(f"  Batches train: {len(train_loader):,}")


## 2. Arquitectura BiLSTM

### Diseño

```
Entrada: (batch_size, max_len)   — índices de tokens
    ↓
Embedding(vocab_size, embed_dim, padding_idx=0)
    ↓  (batch_size, max_len, embed_dim)
BiLSTM(embed_dim → hidden_size×2, bidireccional=True, n_layers=2, dropout)
    ↓  (batch_size, max_len, hidden_size×2)
Global Max Pooling sobre la dimensión temporal
    ↓  (batch_size, hidden_size×2)
Dropout
    ↓
Dense(hidden_size×2 → hidden_size)   + ReLU
    ↓
Dense(hidden_size → N_CLASSES)       + Softmax (implícito en CrossEntropyLoss)
```

### Por qué BiLSTM > LSTM unidireccional

- El LSTM unidireccional solo ve el contexto pasado (izq→der).
- La BiLSTM procesa también de derecha a izquierda, capturando contexto futuro.
- Ejemplo: *"A pesar del **ruido**, el servicio fue increíble"* — la palabra
  "ruido" es modificada por lo que viene después; la BiLSTM lo capta mejor.
- Número de parámetros ≈ 2× el LSTM unidireccional de Felipe → comparación interesante.

### Global Max Pooling

En lugar de usar solo el hidden state del último token (como hace el LSTM simple),
el Global Max Pooling toma el valor máximo de cada dimensión en toda la secuencia.
Esto captura los rasgos más activados a lo largo del texto, independientemente de
su posición — más robusto para reseñas de longitud variable.

In [ ]:
# ── Modelo BiLSTM ─────────────────────────────────────────────────────────────
class BiLSTMClassifier(nn.Module):
    """
    Clasificador de texto basado en una BiLSTM bidireccional de 2 capas.

    Arquitectura:
        Embedding → BiLSTM (2 layers) → Global Max Pooling → Dropout
        → Dense(ReLU) → Dense(logits)

    Args:
        vocab_size  (int)  : Tamaño del vocabulario (incluye <PAD> y <UNK>).
        embed_dim   (int)  : Dimensión de los embeddings de palabras.
        hidden_size (int)  : Tamaño del estado oculto de la LSTM (por dirección).
                             La BiLSTM produce hidden_size×2 en la salida.
        n_layers    (int)  : Número de capas de la BiLSTM (default 2).
        dropout     (float): Dropout aplicado entre capas LSTM y antes del Dense.
        n_classes   (int)  : Número de clases de salida.
        pad_idx     (int)  : Índice del token de padding (default 0).
    """

    def __init__(
        self,
        vocab_size  : int,
        embed_dim   : int   = 128,
        hidden_size : int   = 128,
        n_layers    : int   = 2,
        dropout     : float = 0.4,
        n_classes   : int   = 5,
        pad_idx     : int   = 0,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx,   # el gradient no fluye por los tokens de padding
        )

        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,    # <-- clave: bidireccional
            dropout=dropout if n_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)

        # La BiLSTM produce hidden_size × 2 (una dirección izq→der + der→izq)
        bilstm_out_dim = hidden_size * 2

        # Capa densa intermedia con activación ReLU
        self.fc1 = nn.Linear(bilstm_out_dim, hidden_size)
        self.relu = nn.ReLU()

        # Capa de salida: logits (la softmax la aplica CrossEntropyLoss)
        self.fc2 = nn.Linear(hidden_size, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Args:
            x: Tensor de índices (batch_size, seq_len).

        Returns:
            logits: Tensor (batch_size, n_classes).
        """
        # 1. Embedding: (B, seq_len) → (B, seq_len, embed_dim)
        embedded = self.embedding(x)

        # 2. BiLSTM: (B, seq_len, embed_dim) → (B, seq_len, hidden×2)
        lstm_out, _ = self.bilstm(embedded)

        # 3. Global Max Pooling temporal: (B, seq_len, hidden×2) → (B, hidden×2)
        #    Toma el máximo en cada dimensión a lo largo de la secuencia
        pooled = lstm_out.max(dim=1).values

        # 4. Dropout + Dense + ReLU
        out = self.dropout(pooled)
        out = self.relu(self.fc1(out))

        # 5. Logits finales
        logits = self.fc2(out)
        return logits


# ── Verificar arquitectura ─────────────────────────────────────────────────────
model_test = BiLSTMClassifier(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=128,
                               n_layers=2, dropout=0.4, n_classes=N_CLASSES)
model_test = model_test.to(DEVICE)

# Contar parámetros
n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {n_params:,}")

# Test con un batch ficticio
x_fake = torch.randint(0, VOCAB_SIZE, (4, MAX_LEN)).to(DEVICE)
logits_fake = model_test(x_fake)
print(f"Input shape  : {x_fake.shape}")
print(f"Output shape : {logits_fake.shape}  (debe ser [4, {N_CLASSES}])")
print("✓ Arquitectura verificada correctamente.")
del model_test, x_fake, logits_fake


## 3. Experimento 1: BiLSTM sin class weights (`bilstm_v1`)

**Propósito:** Línea base. Veremos qué métricas obtiene sin compensar el desbalance.
Se espera una accuracy razonable pero F1 macro bajo, especialmente en clases 2 y 3.

In [ ]:
# ── Experimento 1: bilstm_v1 (sin class weights) ─────────────────────────────
EMBED_DIM   = 128
HIDDEN_SIZE = 128
N_LAYERS    = 2
DROPOUT     = 0.4

model_v1 = BiLSTMClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
    n_classes=N_CLASSES,
)

config_v1 = {
    "model_name"      : "bilstm_v1",
    "owner"           : "Yibby",
    "track"           : "PP",
    # Hiperparámetros de entrenamiento
    "n_epochs"        : 20,
    "lr"              : 1e-3,
    "patience"        : 4,
    "weight_decay"    : 1e-5,
    "use_lr_scheduler": True,
    "clip_grad"       : 1.0,
    "checkpoint_path" : str(RESULTS_DIR / "bilstm_v1_best.pt"),
    "seed"            : SEED,
    # Metadatos del modelo
    "embedding_dim"   : EMBED_DIM,
    "hidden_size"     : HIDDEN_SIZE,
    "dropout"         : DROPOUT,
    "use_class_weights": False,
}

print("Entrenando bilstm_v1 (sin class weights)...")
metrics_v1 = train(model_v1, train_loader, val_loader, config_v1)


In [ ]:
# ── Evaluación en test set — bilstm_v1 ───────────────────────────────────────
# IMPORTANTE: Esta evaluación se hace UNA SOLA VEZ, al final.
# Nunca usar el test set para tunear hiperparámetros.

y_true_v1, y_pred_v1 = evaluate(
    model_v1, test_loader,
    checkpoint_path=config_v1["checkpoint_path"],
    config=config_v1,
)

metrics_v1 = finalize_and_save(
    y_true=y_true_v1,
    y_pred=y_pred_v1,
    metrics_dict=metrics_v1,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    class_names=CLASS_NAMES,
    show_plots=True,
)

print(f"\n→ bilstm_v1 | Accuracy: {metrics_v1['metrics']['accuracy']:.4f} | "
      f"F1 macro: {metrics_v1['metrics']['f1_macro']:.4f}")


## 4. Experimento 2: BiLSTM con class weights (`bilstm_v2`)

**Propósito:** Compensar el desbalance del dataset. Se espera que las clases 2 y 3
(bajo F1 en v1) mejoren, aunque la accuracy global puede bajar levemente.
Este es el experimento principal — **es la versión que va en el JSON final** si mejora v1.

In [ ]:
# ── Experimento 2: bilstm_v2 (con class weights) ─────────────────────────────
model_v2 = BiLSTMClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
    n_classes=N_CLASSES,
)

config_v2 = {
    **config_v1,                  # hereda todos los hiperparámetros de v1
    "model_name"       : "bilstm_v2",
    "checkpoint_path"  : str(RESULTS_DIR / "bilstm_v2_best.pt"),
    "class_weights"    : class_weights,   # <-- diferencia clave vs v1
    "use_class_weights": True,
    "seed"             : SEED,
}

print("Entrenando bilstm_v2 (con class weights)...")
metrics_v2 = train(model_v2, train_loader, val_loader, config_v2)


In [ ]:
# ── Evaluación en test set — bilstm_v2 ───────────────────────────────────────
y_true_v2, y_pred_v2 = evaluate(
    model_v2, test_loader,
    checkpoint_path=config_v2["checkpoint_path"],
    config=config_v2,
)

metrics_v2 = finalize_and_save(
    y_true=y_true_v2,
    y_pred=y_pred_v2,
    metrics_dict=metrics_v2,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    class_names=CLASS_NAMES,
    show_plots=True,
)

print(f"\n→ bilstm_v2 | Accuracy: {metrics_v2['metrics']['accuracy']:.4f} | "
      f"F1 macro: {metrics_v2['metrics']['f1_macro']:.4f}")


## 5. Comparación: bilstm_v1 vs bilstm_v2

In [ ]:
# ── Tabla comparativa v1 vs v2 ────────────────────────────────────────────────
import pandas as pd

def resumen_metricas(metrics_dict):
    m = metrics_dict["metrics"]
    t = metrics_dict["training"]
    return {
        "Modelo"           : metrics_dict["model_name"],
        "Class weights"    : metrics_dict["config"]["use_class_weights"],
        "Accuracy"         : f"{m['accuracy']:.4f}",
        "Precision macro"  : f"{m['precision_macro']:.4f}",
        "Recall macro"     : f"{m['recall_macro']:.4f}",
        "F1 macro"         : f"{m['f1_macro']:.4f}",
        "Épocas ejecutadas": t["epochs_run"],
        "Mejor época"      : t["best_epoch"],
        "Tiempo (s)"       : t["training_time_seconds"],
        "# Parámetros"     : f"{metrics_dict['config']['n_params']:,}",
    }

comp_df = pd.DataFrame([resumen_metricas(metrics_v1), resumen_metricas(metrics_v2)])
print("\n" + "="*70)
print("COMPARACIÓN: bilstm_v1 (sin CW) vs bilstm_v2 (con CW)")
print("="*70)
print(comp_df.to_string(index=False))
print()
print("F1 por clase:")
for modelo, metrics in [("bilstm_v1", metrics_v1), ("bilstm_v2", metrics_v2)]:
    f1c = metrics["metrics"]["f1_per_class"]
    print(f"  {modelo}: " + "  ".join([f"cls{k}: {v:.3f}" for k, v in f1c.items()]))
print()
print("Interpretación:")
print("  → Si bilstm_v2 mejora el F1 macro (especialmente clases 2 y 3),")
print("    los class weights son efectivos para manejar el desbalance.")
print("  → La versión con mayor F1 macro se reporta como la versión final.")


## 6. Selección de la versión final y guardado del JSON

In [ ]:
# ── Seleccionar la mejor versión y guardar JSON final ──────────────────────────
import shutil

# Comparar F1 macro de las dos versiones
f1_v1 = metrics_v1["metrics"]["f1_macro"]
f1_v2 = metrics_v2["metrics"]["f1_macro"]

if f1_v2 >= f1_v1:
    best_metrics   = metrics_v2
    best_config    = config_v2
    best_model_key = "bilstm_v2"
    print(f"✓ Mejor versión: bilstm_v2 (F1 macro: {f1_v2:.4f} ≥ bilstm_v1: {f1_v1:.4f})")
else:
    best_metrics   = metrics_v1
    best_config    = config_v1
    best_model_key = "bilstm_v1"
    print(f"✓ Mejor versión: bilstm_v1 (F1 macro: {f1_v1:.4f} > bilstm_v2: {f1_v2:.4f})")

# Renombrar el JSON final como bilstm_metrics.json (nombre esperado por el equipo)
best_json_src = RESULTS_DIR / f"{best_model_key}_metrics.json"
best_json_dst = RESULTS_DIR / "bilstm_metrics.json"
shutil.copy(best_json_src, best_json_dst)
print(f"\n  JSON final guardado → results/bilstm_metrics.json")

# Copiar el checkpoint del mejor modelo
best_ckpt_src = Path(best_config["checkpoint_path"])
best_ckpt_dst = RESULTS_DIR / "bilstm_best.pt"
shutil.copy(best_ckpt_src, best_ckpt_dst)
print(f"  Checkpoint final   → results/bilstm_best.pt")


## 7. Análisis del mejor modelo

### Evaluación por clase

Las clases intermedias (2 y 3 estrellas) son típicamente las más difíciles:
- Tienen menos muestras (desbalance).
- Sus textos son más ambiguos (mezclan críticas y elogios).

El F1 por clase revela si el modelo discrimina bien en todo el espectro de ratings.

In [ ]:
# ── Análisis detallado por clase ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

m = best_metrics["metrics"]
f1_por_clase = m["f1_per_class"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# F1 por clase
ax = axes[0]
clases_names = list(f1_por_clase.keys())
f1_vals = list(f1_por_clase.values())
colors = ["#D32F2F", "#FF7043", "#FDD835", "#66BB6A", "#1E88E5"]
bars = ax.bar(clases_names, f1_vals, color=colors)
ax.set_title(f"F1 por clase — {best_metrics['model_name']}", fontsize=12)
ax.set_xlabel("Clase (estrellas)")
ax.set_ylabel("F1-Score")
ax.set_ylim(0, 1.0)
ax.axhline(y=m["f1_macro"], color="gray", linestyle="--", label=f"F1 macro = {m['f1_macro']:.3f}")
ax.legend()
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=10)

# Resumen de métricas
ax = axes[1]
ax.axis("off")
tabla = [
    ["Métrica", "Valor"],
    ["Accuracy", f"{m['accuracy']:.4f}"],
    ["Precision macro", f"{m['precision_macro']:.4f}"],
    ["Recall macro", f"{m['recall_macro']:.4f}"],
    ["F1 macro", f"{m['f1_macro']:.4f}"],
    ["Épocas ejecutadas", str(best_metrics["training"]["epochs_run"])],
    ["Mejor época", str(best_metrics["training"]["best_epoch"])],
    ["# Parámetros", f"{best_metrics['config']['n_params']:,}"],
]
t = ax.table(cellText=tabla[1:], colLabels=tabla[0],
             cellLoc="center", loc="center")
t.auto_set_font_size(False)
t.set_fontsize(10)
t.scale(1.2, 1.5)
ax.set_title(f"Resumen — {best_metrics['model_name']}", fontsize=12)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "bilstm_analisis_final.png", dpi=150, bbox_inches="tight")
plt.show()
print("  Figura guardada → figures/bilstm_analisis_final.png")


## 8. Párrafo para el artículo IEEE y el reporte técnico

In [ ]:
# ── Generar párrafo descriptivo para los documentos ──────────────────────────
# Este texto va directamente en el artículo IEEE y en el reporte técnico.
# Felipe y Daniel pueden copiarlo en sus documentos.

m = best_metrics["metrics"]
cfg = best_metrics["config"]
t   = best_metrics["training"]

parrafo = f"""
=== PÁRRAFO PARA EL ARTÍCULO IEEE / REPORTE TÉCNICO ===

Modelo Clásico 2 — BiLSTM Bidireccional (Yibby González)

Se implementó una red LSTM bidireccional (BiLSTM) como segundo modelo clásico. La arquitectura
consiste en una capa de embeddings entrenables de dimensión {cfg['embedding_dim']}, seguida de
{2} capas BiLSTM con {cfg['hidden_size']} unidades ocultas por dirección (dimensión de salida: 
{cfg['hidden_size']*2}), Global Max Pooling temporal, una capa densa con activación ReLU y la 
capa de clasificación final. Se aplicó dropout de {cfg['dropout']} para regularización y 
class weights balanceados para compensar el desbalance del dataset.

La BiLSTM procesa cada secuencia en ambas direcciones (izquierda→derecha y derecha→izquierda),
lo que le permite capturar tanto el contexto pasado como el futuro para cada token. El Global 
Max Pooling agrega la información de toda la secuencia tomando el valor máximo en cada dimensión,
haciéndolo robusto a la longitud variable de las reseñas.

Resultados en el conjunto de prueba:
  - Accuracy: {m['accuracy']:.4f}
  - F1 macro: {m['f1_macro']:.4f}
  - Precision macro: {m['precision_macro']:.4f}
  - Recall macro: {m['recall_macro']:.4f}
  - Parámetros entrenables: {cfg['n_params']:,}
  - Épocas ejecutadas: {t['epochs_run']} (mejor: época {t['best_epoch']})
  - Tiempo de entrenamiento: {t['training_time_seconds']}s

F1 por clase: {m['f1_per_class']}
"""

print(parrafo)


## 9. Checklist de entregables

In [ ]:
# ── Checklist de entregables del HITO 2 ──────────────────────────────────────
from pathlib import Path

entregables = [
    (RESULTS_DIR / "bilstm_metrics.json",  "results/bilstm_metrics.json  [JSON estandarizado]"),
    (RESULTS_DIR / "bilstm_best.pt",       "results/bilstm_best.pt       [Checkpoint mejor modelo]"),
    (FIGURES_DIR / "bilstm_v1_curves.png", "figures/bilstm_v1_curves.png [Curvas v1]"),
    (FIGURES_DIR / "bilstm_v2_curves.png", "figures/bilstm_v2_curves.png [Curvas v2]"),
    (FIGURES_DIR / "bilstm_v1_confusion_matrix.png", "figures/bilstm_v1_confusion_matrix.png"),
    (FIGURES_DIR / "bilstm_v2_confusion_matrix.png", "figures/bilstm_v2_confusion_matrix.png"),
]

print("Checklist HITO 2 — Yibby González:")
all_ok = True
for path, label in entregables:
    status = "✓" if path.exists() else "✗ FALTA"
    if not path.exists():
        all_ok = False
    print(f"  [{status}] {label}")

print()
if all_ok:
    print("✅ Todos los entregables listos. Listo para el HITO 2.")
else:
    print("⚠ Algunos entregables faltan. Ejecutar las celdas anteriores.")
